In [1]:
# PROJECT SETUP
# set DRAFT = False for final output; DRAFT = True is faster
DRAFT = True

import itertools
import logging
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings( "ignore", module = "plotnine\..*" )

import cmdstanpy as csp
csp.utils.get_logger().setLevel(logging.ERROR)

import numpy as np
import statistics as stat
import pandas as pd
import plotnine as pn
import time

class StopWatch:
    def __init__(self):
        self.start()
    def start(self):
        self.start_time = time.time()
    def elapsed(self):
        return time.time() - self.start_time
timer = StopWatch()

<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_6829/1528855002.py:9: SyntaxWarning: invalid escape sequence '\.'
  warnings.filterwarnings( "ignore", module = "plotnine\..*" )
/home/kaiserd/projects/HETKER/proj_HETKER/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

# Sampling infection kernels

In [45]:
# Declare constants
data = {
    # Population constants
    'V': 2,
    'kmax': 2,
    # Hyperparameters
    'AVERAGE_SCALE': 2,
    'DEVIATION_SCALE': 4,
}

# Load STAN model
sampling = csp.CmdStanModel(stan_file = '../src/sampling.stan')

# Sample infection kernels
sample = sampling.sample(
    data          = data,
    # seed          = 37,
    chains        = 20,
    iter_sampling = 100,
    iter_warmup   = 100,
    show_progress = True,
    show_console  = False
)

chain 1 |          | 00:00 Status






































chain 10 |          | 00:00 Status






































































































































chain 1 |██████████| 00:00 Sampling completed


















chain 1 |██████████| 00:00 Sampling completed
chain 2 |██████████| 00:00 Sampling completed
chain 3 |██████████| 00:00 Sampling completed
chain 4 |██████████| 00:00 Sampling completed
chain 5 |██████████| 00:00 Sampling completed
chain 6 |██████████| 00:00 Sampling completed
chain 7 |██████████| 00:00 Sampling completed
chain 8 |██████████| 00:00 Sampling completed
chain 9 |██████████| 00:00 Sampling completed
chain 10 |██████████| 00:00 Sampling completed
chain 11 |██████████| 00:00 Sampling completed
chain 12 |██████████| 00:00 Sampling completed
chain 13 |██████████| 00:00 Sampling completed
chain 14 |██████████| 00:00 Sampling completed
chain 15 |██████████| 00:00 Sampling completed
chain 16 |██████████| 00:00 Sampling completed
chain 17 |██████████| 00

In [47]:
sample.stan_variables()['c'][0]

array([[0.38137061, 0.64730833],
       [0.58817858, 0.7177967 ]])

# Inference from data

In [2]:
# Declare constants
data = {
    # Population constants
    'V': 2,
    'kmax': 2,
    'K': 2,
    # Hyperparameters
    'AVERAGE_SCALE': [2, 2],
    'DEVIATION_SCALE': [1, 2],
}

# Load state data
M = [
    [0, 0],
    [1, 0]
]
N = [
    [0, 0],
    [0, 0]
]

data['M'] = M
data['N'] = N

# Load STAN model
model = csp.CmdStanModel(
    stan_file = '../src/model_mixture.stan',
    # exe_file='../target/build/'
)

# Infer kernels from infection data
sample = model.sample(
    data          = data,
    seed          = 37,
    chains        = 1,
    iter_sampling = 1,
    iter_warmup   = 20,
    show_progress = False,
    show_console  = True
)

Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1
Chain [1] num_warmup = 20
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.8 (Default)
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpcvq7hl5z/axlya949.json
Chain [1] init = 2 (Default)
Chain [1] random
Chain [1] seed = 37
Chain [1] output
Chain [1] file = 

In [3]:
sample.stan_variables()

{'mu': array([[[0.45907478, 0.54092522],
         [0.46587884, 0.53412116]]]),
 'sigma': array([[0.52164943, 0.09818194]]),
 'alpha_k': array([[[0.23144813, 0.59403072],
         [0.57536408, 0.56303866]]]),
 'pi': array([[[0.58344684, 0.41655316],
         [0.22426336, 0.77573664]]]),
 'alpha': array([[[0.37470741, 0.49823633],
         [0.        , 0.56998904]]]),
 'c': array([[[0.59259596, 0.62204477],
         [0.5       , 0.63876065]]])}